In [ ]:
# Install dependencies

%pip install anthropic python-dotenv

In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"


In [4]:
# Make a request

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)
response
response.content[0].text

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentance"
        }
    ]
)
response.content[0].text

"Here's a sentence for you:\n\nThe old lighthouse stood silently on the rocky cliff, its beam cutting through the foggy night to guide ships safely to shore."

In [5]:
# Client - Api interface

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
         "temperature": temperature, 
         "stop_sequences":stop_sequences
    }
     
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
# Multi-Turn conversations

# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

print("Prompt 1: "+ answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

print("Prompt 2: " + final_answer)

Prompt 1: Quantum computing is a revolutionary computing paradigm that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.
Prompt 2: Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are limited to either 0 or 1, enabling them to perform many calculations in parallel.


In [8]:
# System Prompts

# Without system prompt
answer = chat(messages)

print(answer)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)

Unlike classical computers that use bits representing either 0 or 1, quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, enabling them to perform many calculations in parallel.
I notice you've moved away from math problems! Since I'm here to help you learn math step by step, would you like to work on a specific math question or concept instead? 

What math topic are you currently studying or what problem would you like help solving?


In [9]:
# Temperature
# Low temperature - more predictable
answer = chat(messages, temperature=0.0)

# High temperature - more creative  
answer = chat(messages, temperature=1.0)


In [10]:
# Controlling model output
messages = []
add_user_message(messages, "Is tea or coffee better at breakfast?")
add_assistant_message(messages, "Coffee is better because")
answer = chat(messages)

answer

" of the higher caffeine content, which can help wake you up in the morning.\n\nHowever, tea can be a good alternative if you're sensitive to caffeine or prefer a milder start to your day. Some teas, particularly green tea, also provide antioxidants and may offer health benefits.\n\nConsider your personal caffeine tolerance, taste preferences, and how you want to feel in the morning. Many people also enjoy both, just at different times of the day."

In [11]:
# Stop Sequances

messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=[", 5"])

answer

'1, 2, 3, 4'

In [12]:
# Structured Data

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages)

add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ],\n  "State": "ENABLED"\n}\n'

In [13]:
import json 

json_format = json.loads(text)

json_format

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}],
 'State': 'ENABLED'}

Exercise! 

Use message prefilling and stop seqences only to get three different commands in a single response
There shouldn't be any comments or explanation
Hint: message prefilling isn't limited to just characters like ```

In [14]:

messages = []

prompt = """ 
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

add_assistant_message(messages, "Here are all three commands in a single block without any comments: \n```bash")

text = chat(messages, stop_sequences=["```"])

print(text)




aws s3 ls
aws ec2 describe-instances
aws iam list-users



In [15]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls
aws ec2 describe-instances
aws iam list-users
